In [1]:
!pip install -q transformers peft accelerate bitsandbytes timm
!pip install scikit-learn peft tqdm torch

In [2]:
import torch
import torch.nn as nn
import numpy as np

from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader, random_split

from transformers import (
    AutoModel,
    AutoImageProcessor
)

from peft import (
    LoraConfig,
    get_peft_model
)

from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score

device = "cuda" if torch.cuda.is_available() else "cpu"

print(device)

cuda


In [3]:
MODEL_NAME = "facebook/ijepa_vith14_1k"

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=processor.image_mean,
        std=processor.image_std
    )
])

full_train = CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=train_transform
)

test_dataset = CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=test_transform
)

train_size = 45000
val_size = 5000

train_dataset, val_dataset = random_split(
    full_train,
    [train_size, val_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2
)

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

100%|██████████| 170M/170M [23:45<00:00, 120kB/s]    


In [13]:
class IJEPAClassifier(nn.Module):

    def __init__(
        self,
        backbone,
        num_classes=10
    ):
        super().__init__()

        self.backbone = backbone

        hidden_size = backbone.config.hidden_size

        self.classifier = nn.Linear(
            hidden_size,
            num_classes
        )

    def forward(self, pixel_values):

        outputs = self.backbone(
            pixel_values=pixel_values
        )

        features = outputs.last_hidden_state.mean(dim=1)

        logits = self.classifier(features)

        return logits

In [5]:
backbone = AutoModel.from_pretrained(
    MODEL_NAME
)

for param in backbone.parameters():
    param.requires_grad = False

linear_probe_model = IJEPAClassifier(
    backbone
).to(device)

config.json:   0%|          | 0.00/497 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

In [6]:
backbone_lora = AutoModel.from_pretrained(
    MODEL_NAME
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", 
        "k_proj", 
        "v_proj"
    ]
)

backbone_lora = get_peft_model(
    backbone_lora,
    lora_config
)
for name, param in backbone_lora.named_parameters():

    if "lora" not in name:
        param.requires_grad = False

backbone_lora.print_trainable_parameters()

lora_model = IJEPAClassifier(
    backbone_lora
).to(device)

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

trainable params: 3,932,160 || all params: 634,694,400 || trainable%: 0.6195


In [7]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    running_loss = 0

    preds = []
    labels_all = []

    for images, labels in tqdm(loader):

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        pred = outputs.argmax(1)

        preds.extend(
            pred.cpu().numpy()
        )

        labels_all.extend(
            labels.cpu().numpy()
        )

    acc = accuracy_score(
        labels_all,
        preds
    )

    return running_loss / len(loader), acc

In [8]:
@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    running_loss = 0

    preds = []
    labels_all = []

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        running_loss += loss.item()

        pred = outputs.argmax(1)

        preds.extend(
            pred.cpu().numpy()
        )

        labels_all.extend(
            labels.cpu().numpy()
        )

    acc = accuracy_score(
        labels_all,
        preds
    )

    return running_loss / len(loader), acc

In [9]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    linear_probe_model.parameters(),
    lr=1e-3
)

EPOCHS = 5

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        linear_probe_model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc = evaluate(
        linear_probe_model,
        val_loader,
        criterion
    )

    print(
        f"Epoch {epoch+1}"
        f" | Train Acc={train_acc:.4f}"
        f" | Val Acc={val_acc:.4f}"
    )

  0%|          | 0/352 [00:00<?, ?it/s]

Epoch 1 | Train Acc=0.8128 | Val Acc=0.8644


  0%|          | 0/352 [00:00<?, ?it/s]

Epoch 2 | Train Acc=0.8792 | Val Acc=0.8846


  0%|          | 0/352 [00:00<?, ?it/s]

Epoch 3 | Train Acc=0.8945 | Val Acc=0.8966


  0%|          | 0/352 [00:00<?, ?it/s]

Epoch 4 | Train Acc=0.9004 | Val Acc=0.9032


  0%|          | 0/352 [00:00<?, ?it/s]

Epoch 5 | Train Acc=0.9078 | Val Acc=0.9054


In [10]:
test_loss, test_acc = evaluate(
    linear_probe_model,
    test_loader,
    criterion
)

print("Linear Probe Test Accuracy:", test_acc)

Linear Probe Test Accuracy: 0.9044


In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    lora_model.parameters(),
    lr=2e-4
)

EPOCHS = 5

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch(
        lora_model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc = evaluate(
        lora_model,
        val_loader,
        criterion
    )

    print(
        f"Epoch {epoch+1}"
        f" | Train Acc={train_acc:.4f}"
        f" | Val Acc={val_acc:.4f}"
    )

  0%|          | 0/352 [00:00<?, ?it/s]

Epoch 1 | Train Acc=0.9442 | Val Acc=0.9838


  0%|          | 0/352 [00:00<?, ?it/s]

Epoch 2 | Train Acc=0.9906 | Val Acc=0.9884


  0%|          | 0/352 [00:00<?, ?it/s]

Epoch 3 | Train Acc=0.9947 | Val Acc=0.9880


  0%|          | 0/352 [00:00<?, ?it/s]

Epoch 4 | Train Acc=0.9957 | Val Acc=0.9912


  0%|          | 0/352 [00:00<?, ?it/s]

Epoch 5 | Train Acc=0.9975 | Val Acc=0.9884


In [12]:
test_loss, test_acc = evaluate(
    lora_model,
    test_loader,
    criterion
)

print("LoRA Test Accuracy:", test_acc)

LoRA Test Accuracy: 0.9869
